In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException, TimeoutException
import pandas as pd
import time


url = 'https://www.tgju.org/profile/geram18/history'

# مسیر درایور Chrome (خودت تنظیم کن)
service = Service('/path/to/chromedriver')  # مثلاً '/usr/local/bin/chromedriver'

# گزینه‌های Chrome (برای headless کردن، اگر نخوای مرورگر باز بشه: options.add_argument('--headless'))
options = webdriver.ChromeOptions()
driver = webdriver.Chrome(service=service, options=options)

# باز کردن صفحه
driver.get(url)

# DataFrame اصلی برای ذخیره همه داده‌ها
all_data = pd.DataFrame()

# لوپ برای استخراج صفحات
while True:
    try:
        # صبر تا لود کامل جدول (حداکثر 10 ثانیه)
        WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.TAG_NAME, 'table')))  # اگر جدول با تگ table باشه
        
        # استخراج جدول از page_source (فرض کنیم اولین جدول در صفحه index 0 هست – inspect کن و تغییر بده اگر چند جدول باشه)
        df_page = pd.read_html(driver.page_source)[0]
        
        # اپند کردن به DataFrame اصلی
        all_data = pd.concat([all_data, df_page], ignore_index=True)
        
        print(f"صفحه استخراج شد: {len(df_page)} ردیف")
        
    except TimeoutException:
        print("جدول لود نشد – چک کن selectorها رو.")
        break
    except Exception as e:
        print(f"خطا در استخراج جدول: {e}")
        break
    
    try:
        # پیدا کردن و کلیک روی دکمه بعدی (فرض کلاس 'next' – inspect عنصر و تغییر بده به By.XPATH یا By.ID اگر لازم)
        next_button = WebDriverWait(driver, 10).until(EC.element_to_be_clickable((By.CLASS_NAME, 'next')))
        next_button.click()
        
        # 3 ثانیه دیلی (طبق درخواستیت)
        time.sleep(3)
        
    except (NoSuchElementException, TimeoutException):
        print("دکمه بعدی وجود نداره – پایان صفحات.")
        break
    except Exception as e:
        print(f"خطا در کلیک بعدی: {e}")
        break

# بستن درایور
driver.quit()

# ذخیره DataFrame به CSV (یا هر فرمت دیگه)
all_data.to_csv('gold_data.csv', index=False, encoding='utf-8-sig')  # utf-8-sig برای پشتیبانی پارسی
print("داده‌ها ذخیره شد در gold_data.csv")

# نمایش نمونه داده‌ها (برای تست)
print(all_data.head())